# patgen — mining templates from messy bilingual SMS

This notebook walks the whole approach on a synthetic corpus that mixes
finance, e-commerce shipping, travel bookings, support, 2FA and log alerts:
Arabic + English, Arabic-Indic digits, tashkeel, bidi marks, cp1252 mojibake
in the middle of otherwise clean text, and truncated tails.

1. the data and what is wrong with it
2. normalization + mojibake repair
3. tokenization + entity masking (the step that makes Drain work here)
4. Drain-style clustering and wildcard refinement
5. the learned library: templates, typed slots, coverage
6. production matching + throughput
7. what stays unmatched, and tuning

In [1]:
import random
import re
import sys
import time
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

from patgen import (  # noqa: E402
    DrainTree,
    LearnConfig,
    TemplateMatcher,
    learn_templates,
    normalize,
    prepare,
    repair_mojibake,
    tokenize,
)
from patgen.entities import mask_tokens  # noqa: E402
from patgen.io_csv import read_texts  # noqa: E402
from patgen.report import coverage_report  # noqa: E402

CSV = Path.cwd().parent / "examples" / "sms_sample.csv"
messages = list(read_texts([CSV]))
len(messages), messages[0]

(5000,
 'سحب  نقدي 35,537.14 AED من الحساب ***138 بتاريخ 11/12/2024 الرصيد المتاح 24,190.80')

## 1. What the raw data looks like

In [2]:
random.seed(7)
for m in random.sample(messages, 8):
    print(repr(m))

'Booking 546732529 confirmed. Flight QR890 from جده to جده on 2024-05-16 at 13:59. Passenger خالد العتيبي'
'Purchase of AED 23,700.29 at هنقرستيشن on card ending xxxx1584 on 2024-04-08 03:24. Available balance AED 248,263.26'
'Your order 363173831 has been shipped. Track at https://noon.com/orders Estimate'
'تم  تحويل مبلغ 20,037.51 SAR الى نورة الشمري رقم المرجع 170914703 الرصيد 443,977.37'
'Booking ٦٣٨٨٧٩٨٥٣ confirmed. Flight QR٨٩٠ from cairo to dubai on ٢٥/٠٤/٢٠٢٤ at ٠٠:٥٧. Passenger sara a.'
'\u200fYour order 316253400 has been confirmed. Track at https://support.example.com/ticket/xyz Estimated delivery 2024-06-16\u200e'
'رمز التحقق الخاص بك هو 392697 صالح لمدة 10 دقائق لا تشاركه مع احد'
'Dear customer, service المدفوعات الدولية was activated on account ***982. Call 966554785822 for help'


Three separate problems in one column:

* **two scripts** — Arabic and English templates, often mixed inside a message;
* **encoding damage** — `Ø±.Ø³` is `ر.س` that went through UTF-8 → cp1252,
  and it is usually only *part* of the message, so a whole-string round trip
  cannot fix it;
* **noise** — Arabic-Indic digits, tashkeel, bidi marks, double spaces,
  truncated tails.

In [3]:
broken = [m for m in messages if re.search("[ÂÃØÙÚÛ][\u0080-\u00ff\u0152-\u0178]", m)]
print(f"{len(broken)}/{len(messages)} messages carry mojibake\n")
for m in broken[:3]:
    print("raw :", m)
    print("fixed:", repair_mojibake(m), "\n")

231/5000 messages carry mojibake

raw : Bill payment Ø±.Ø³ 331.03 to Ù‡Ù†Ù‚Ø±Ø³ØªÙŠØ´Ù† succeeded. Fee ر.س 33.56. Ref 296449540
fixed: Bill payment ر.س 331.03 to هنقرستيشن succeeded. Fee ر.س 33.56. Ref 296449540 

raw : خصم Ø±Ø³ÙˆÙ… 46.75 AED Ø¹Ù„Ù‰ Ø§Ù„Ø­Ø³Ø§Ø¨ ***580 رقم العملية 974712306
fixed: خصم رسوم 46.75 AED على الحساب ***580 رقم العملية 974712306 

raw : تم ØªØ£ÙƒÙŠØ¯ Ø­Ø¬Ø²Ùƒ 583010316 Ø±Ø­Ù„Ø© TK123 Ù…Ù† dammam الى جده بتاريخ 07/10/2024 الساعه 13:26
fixed: تم تأكيد حجزك 583010316 رحلة TK123 Ù…Ù† dammam الى جده بتاريخ 07/10/2024 الساعه 13:26 



## 2. Normalization

In [4]:
samples = [
    "‏سحب نقدي ٦٩,٩٥١.٧٠ ر.س من الحساب ***815‎",
    "Bill payment Ø±.Ø³ 37,955.74 to ACME",
    "رَصيدُك الحالي ٢٥٠٫٥٠ ريال",
    "عملية شراء لدى صيدلية النهدي",
]
for s in samples:
    print(f"{s!r}\n  -> {normalize(s)!r}")

'\u200fسحب نقدي ٦٩,٩٥١.٧٠ ر.س من الحساب ***815\u200e'
  -> 'سحب نقدي 69,951.70 ر.س من الحساب ***815'
'Bill payment Ø±.Ø³ 37,955.74 to ACME'
  -> 'bill payment ر.س 37,955.74 to acme'
'رَصيدُك الحالي ٢٥٠٫٥٠ ريال'
  -> 'رصيدك الحالي 250.50 ريال'
'عملية شراء لدى صيدلية النهدي'
  -> 'عمليه شراء لدي صيدليه النهدي'


Folding (`أإآ→ا`, `ة→ه`, `ى→ي`), digit conversion and mojibake repair all
happen *before* clustering, so the same sentence written four different ways
collapses onto one template instead of four.

## 3. Tokenization + entity masking

In [5]:
msg = "Purchase of ر.س 4,467.79 at starbucks on card ending ****5744 on 29/07/2024 02:48. Available balance ر.س 110,579.86"
prepared = prepare(msg)
print("canonical:", prepared.canonical, "\n")
for m in mask_tokens(tokenize(prepared.canonical)):
    flag = "  <-- entity" if m.is_entity else ""
    print(f"{m.key:<12} {m.value}{flag}")

canonical: purchase of ر . س 4,467.79 at starbucks on card ending * * * * 5744 on 29/07/2024 02:48 . available balance ر . س 110,579.86 

purchase     purchase
of           of
<CURRENCY>   ر . س  <-- entity
<AMOUNT>     4,467.79  <-- entity
at           at
starbucks    starbucks
on           on
card         card
ending       ending
<CARD>       * * * * 5744  <-- entity
on           on
<DATETIME>   29/07/2024 02:48  <-- entity
.            .
available    available
balance      balance
<CURRENCY>   ر . س  <-- entity
<AMOUNT>     110,579.86  <-- entity


This is the key difference from vanilla Drain3. Values are replaced by
**typed** placeholders before clustering, so amounts, dates and card tails
never split a cluster, and the placeholder type is carried into the template
(a plain `<*>` would lose it).

## 4. Clustering, and why wildcards get refined

In [6]:
tree = DrainTree()
for m in messages[:2000]:
    tree.add(prepare(m).keys)
print(f"{len(tree.clusters)} raw clusters from 2000 messages")
for c in sorted(tree.clusters, key=lambda c: -c.count)[:6]:
    print(f"{c.count:>5}  {' '.join(c.tokens)}")

164 raw clusters from 2000 messages
   94  your verification code is <NUM> . login from ip <REF> at <TIME> <*>
   90  خصم رسوم <AMOUNT> <CURRENCY> علي الحساب <CARD> رقم <*>
   81  salary of <CURRENCY> <AMOUNT> credited to account <CARD> <*>
   75  تم تحويل مبلغ <AMOUNT> <CURRENCY> الي <*>
   74  atm withdrawal <CURRENCY> <AMOUNT> from account <CARD> on <DATE> . available <*>
   69  تم ايداع راتب بمبلغ <AMOUNT> <CURRENCY> <*> الحساب <CARD> بتاريخ <DATE>


A single truncated message widens a cluster into `<*>`, swallowing fields
that were perfectly extractable. The learner replays every wildcard against the
messages that filled it and splices back the dominant filling — or the dominant
prefix/suffix around the part that genuinely varies — marking it optional when
some messages left it empty.

In [7]:
t0 = time.perf_counter()
library = learn_templates(messages)
print(f"{len(messages)} messages -> {len(library.templates)} templates in {time.perf_counter()-t0:.2f}s")
for t in library.templates[:10]:
    print(f"{t.count:>5}  {t.text}")

5000 messages -> 67 templates in 0.78s
  240  عمليه شراء بمبلغ <AMOUNT:بمبلغ> <CURRENCY:currency> لدي <TEXT:text>?
  227  your card <CARD:card> was declined at <TEXT:text>?
  226  salary of <CURRENCY:currency> <AMOUNT:salary> credited to account <CARD:account> on <DATE:date>
  222  عزيزنا العميل تم <TEXT:العميل>?
  217  your verification code is <NUM:code> . login from ip <REF:ip> at <TIME:time> <TEXT:text>?
  216  رمز التحقق الخاص بك هو <NUM:الخاص> صالح لمده <NUM:لمده> دقايق لا تشاركه مع احد
  211  transfer of <CURRENCY:currency> <AMOUNT:transfer> to <TEXT:text>?
  208  purchase of <CURRENCY:currency> <AMOUNT:purchase> at <TEXT:text> on card ending <CARD:ending> on <DATETIME:datetime> . available balance <CURRENCY:currency_2> <AMOUNT:balance>
  202  طلبيتك <NUM:طلبيتك> في الطريق الي <TEXT:الي>?
  202  alert host <TEXT:host>


## 5. Typed slots per template

In [8]:
t = library.templates[1]
print(t.text, "\n")
for slot in t.slots:
    print(f"{slot.key:<12} {slot.entity:<9} distinct={slot.cardinality:<5} e.g. {slot.examples[:3]}")

your card <CARD:card> was declined at <TEXT:text>? 

card         CARD      distinct=0     e.g. []
text         TEXT      distinct=0     e.g. []


In [9]:
counts = Counter(slot.key for tpl in library.templates for slot in tpl.slots)
counts.most_common(15)

[('currency', 29),
 ('date', 17),
 ('text', 16),
 ('ticket', 10),
 ('account', 9),
 ('purchase', 9),
 ('ending', 9),
 ('datetime', 9),
 ('currency_2', 9),
 ('الحساب', 9),
 ('balance', 8),
 ('time', 6),
 ('الي', 6),
 ('نقدي', 5),
 ('phone', 5)]

## 6. Production matching

In [10]:
matcher = TemplateMatcher(library)
for m in random.sample(messages, 5):
    r = matcher.match(m)
    print(m)
    if r is None:
        print("   -> no match\n")
    else:
        print("   ->", r.template_id, r.entities, "\n")

Your  verification code is ٢٠١٩٤٧. Login from ip ١٠.١٧٠.١٨.١٨٢ at ١٧:٣٣
   -> no match

Dear customer, service apple pay was activated on account ***555. Call 966587754534 for help
   -> t00101 {'service': 'apple pay', 'account': '***555', 'phone': '966587754534'} 

ALERT host worker-03 cpu 97% at 16:49 service الحوالات الفورية status delivered
   -> t00109 {'host': 'worker - 03 cpu 97 % at 16:49 service الحوالات الفوريه status delivered'} 

Your card ****٧٢٤٤ was declined at صيدلية النهدي due to insufficient funds
   -> t00044 {'card': '****7244', 'text': 'صيدليه النهدي due to insufficient funds'} 

تم تأكيد حجزك 492874927 رحلة TK123 من dammam الى riyadh بتاريخ 27/08/2024 الساعه 03:56
   -> t00021 {'حجزك': '492874927', 'رحله': 'tk 123 من dammam الي riyadh', 'date': '27/08/2024', 'time': '03:56'} 



In [11]:
batch = messages * 3
t0 = time.perf_counter()
hits = sum(1 for r in matcher.match_many(batch) if r)
elapsed = time.perf_counter() - t0
print(f"{len(batch)} messages in {elapsed:.2f}s = {len(batch)/elapsed:,.0f} msg/s, {hits/len(batch):.1%} matched")

15000 messages in 1.48s = 10,115 msg/s, 91.8% matched


One normalization pass, then a bucket lookup on the first literal token and
a handful of alternation regexes with named groups — the inner loop runs in the
C regex engine, not in Python.

## 7. What is left unmatched, and tuning

In [12]:
stats = coverage_report(library, messages)
print(f"coverage {stats['coverage']:.1%}")
for m in stats["unmatched_samples"][:8]:
    print(" ", m)

coverage 91.8%
  Booking 774237069 confirmed. Flight SV102 from cairo to الرياض on 04/04/2024 at 20:56. 
  Your verification code is 243246. Login from ip 10.79.126.186 at 22:24
  Your verification code is 659796. Login from ip 10.209.93.229 at 21:00 شكرا لك
  Bill payment SAR 60,162.87 to نون succeeded. Fee SAR 79.02. Ref 896621877 thank you
  Your verification code is 463825. Login from ip 10.71.228.246 at 08:18
  Dear customer, service الحوالات الفورية was activated on account ***945. Call 9
  Your verification code is 707407. Login from ip 10.128.192.27 at 12:45
  Your verification code is 649280. Login from ip 10.208.16.23 at 05:21


The residue is genuinely damaged traffic: messages cut mid-word by the SMS
gateway, or mojibake where one byte of the UTF-8 pair was dropped and nothing
can decode it back. Those are the rows worth alerting on, not templating.

In [13]:
for sim in (0.3, 0.4, 0.5, 0.6, 0.7):
    lib = learn_templates(messages, LearnConfig(sim_threshold=sim))
    cov = coverage_report(lib, messages)["coverage"]
    print(f"sim={sim:<4} templates={len(lib.templates):<5} coverage={cov:.1%}")

sim=0.3  templates=66    coverage=91.8%


sim=0.4  templates=69    coverage=92.4%


sim=0.5  templates=67    coverage=91.8%


sim=0.6  templates=84    coverage=88.7%


sim=0.7  templates=83    coverage=86.9%


Higher similarity = more, tighter templates; lower = fewer, more generic
ones. `--min-support` then trims the long tail before the model is saved with
`library.save("model.json")` and loaded by the serving process.